In [ ]:
# --- 1. SETUP & DATA LOADING ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load your dataset
df = pd.read_csv('C:\\Users\\ibrah\\Downloads\\energydata_complete.csv')

# --- 2. EXPLORATORY DATA ANALYSIS & PREPROCESSING ---
print(f"Dataset shape: {df.shape}")
print(df.info())
print(df.describe())

# Convert date and extract time features
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek

# Define features (X) and target (y)
# 'rv1' and 'rv2' are random variables; you can choose to drop them.
X = df.drop(columns=['Appliances', 'date', 'rv1', 'rv2'])
y = df['Appliances']

# Train-test split (use shuffle=False if respecting time series order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Standardize features (CRITICAL for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train_scaled.shape[0]}, Features: {X_train_scaled.shape[1]}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} Wh")

# --- 3. BASELINE NEURAL NETWORK MODEL ---
def create_baseline_model(input_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # Output layer for regression
    ])
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    return model

baseline_model = create_baseline_model(X_train_scaled.shape[1])
baseline_model.summary()

# Train the baseline model
print("\nTraining baseline model...")
history_baseline = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test_scaled).flatten()
test_rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
test_r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"\n--- Baseline Model Results ---")
print(f"Test RMSE: {test_rmse_baseline:.2f} Wh")
print(f"Test R²: {test_r2_baseline:.3f}")

# --- 4. HYPERPARAMETER TUNING & MODEL IMPROVEMENT ---
# Example: Testing different architectures
def build_model(hidden_layers, neurons_per_layer, activation='relu'):
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(neurons_per_layer, activation=activation))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Example configurations to test (for deeper analysis)
configs = [
    {'hidden_layers': 2, 'neurons': 64},
    {'hidden_layers': 3, 'neurons': 128},
    {'hidden_layers': 4, 'neurons': 64}
]

tuning_results = []
for config in configs:
    print(f"\nTesting config: {config}")
    model = build_model(config['hidden_layers'], config['neurons'])
    history = model.fit(
        X_train_scaled, y_train,
        validation_split=0.15,
        epochs=30,
        batch_size=32,
        verbose=0
    )
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    tuning_results.append({**config, 'test_rmse': rmse})

# Display tuning results
results_df = pd.DataFrame(tuning_results)
print("\n--- Hyperparameter Tuning Results ---")
print(results_df.sort_values('test_rmse'))

# --- 5. FINAL MODEL & EVALUATION ---
# Based on your tuning, define your final architecture
final_model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("\nTraining final model...")
history_final = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    verbose=1,
    callbacks=[
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]
)

# Final evaluation
y_train_pred_final = final_model.predict(X_train_scaled).flatten()
y_test_pred_final = final_model.predict(X_test_scaled).flatten()

train_rmse_final = np.sqrt(mean_squared_error(y_train, y_train_pred_final))
test_rmse_final = np.sqrt(mean_squared_error(y_test, y_test_pred_final))
train_r2_final = r2_score(y_train, y_train_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE")
print("="*50)
print(f"Training RMSE: {train_rmse_final:.2f} Wh | R²: {train_r2_final:.3f}")
print(f"Test RMSE:     {test_rmse_final:.2f} Wh | R²: {test_r2_final:.3f}")

# --- 6. VISUALIZATION & SAVING RESULTS ---
# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_final.history['loss'], label='Training Loss')
plt.plot(history_final.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Model Loss During Training')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot predictions vs actual
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred_final, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Energy (Wh)')
plt.ylabel('Predicted Energy (Wh)')
plt.title('Actual vs Predicted Values')
plt.tight_layout()
plt.show()

# Save your model and results
final_model.save('final_neural_network_model.h5')
print("\nModel saved as 'final_neural_network_model.h5'")

# Save performance metrics to CSV
metrics_df = pd.DataFrame({
    'model': ['Neural Network'],
    'test_rmse': [test_rmse_final],
    'test_r2': [test_r2_final],
    'parameters': [final_model.count_params()]
})
metrics_df.to_csv('nn_performance_metrics.csv', index=False)
print("Performance metrics saved to 'nn_performance_metrics.csv'")

Dataset shape: (19735, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 

C:\Users\ibrah\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,905 (15.25 KB)

 Trainable params: 3,905 (15.25 KB)

 Non-trainable params: 0 (0.00 B)


Training baseline model...
Epoch 1/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 13006.4814 - mae: 65.4263 - val_loss: 9434.6475 - val_mae: 56.3981
Epoch 2/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 9770.0781 - mae: 55.1628 - val_loss: 8614.5439 - val_mae: 53.2022
Epoch 3/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9284.8330 - mae: 53.6872 - val_loss: 8279.6211 - val_mae: 51.9943
Epoch 4/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9040.5117 - mae: 52.9450 - val_loss: 8113.0054 - val_mae: 51.4086
Epoch 5/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8897.7070 - mae: 52.4505 - val_loss: 8015.5703 - val_mae: 51.0752
Epoch 6/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8799.3994 - mae: 52.0713 - val_loss: 7944.6152 - val_mae: 50.8139
Epoch 7/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8723.7236 - mae: 51.7515 - val_loss: 7890.4541 - val_mae: 50.5928
Epoch 8/50
420/420 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8660.1318 - mae: 51.4666 - va

C:\Users\ibrah\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


420/420 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 11241.6426 - mae: 59.8669 - val_loss: 8641.4893 - val_mae: 54.8772 - learning_rate: 0.0010
Epoch 2/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 9207.9121 - mae: 53.4082 - val_loss: 8137.9238 - val_mae: 52.5232 - learning_rate: 0.0010
Epoch 3/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8878.0137 - mae: 52.3144 - val_loss: 7931.9536 - val_mae: 51.5458 - learning_rate: 0.0010
Epoch 4/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 8666.1436 - mae: 51.4407 - val_loss: 7781.8589 - val_mae: 50.9310 - learning_rate: 0.0010
Epoch 5/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 8458.9434 - mae: 50.6826 - val_loss: 7647.3423 - val_mae: 50.7618 - learning_rate: 0.0010
Epoch 6/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8277.2441 - mae: 50.0572 - val_loss: 7544.9727 - val_mae: 50.5513 - learning_rate: 0.0010
Epoch 7/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 8121.3198 - mae: 49.4491 - val_loss: 7455.47